# Phase 0 — Lesson 11: Linux for AI
> Most AI runs on Linux. You need to know enough to not be stuck.

**Source:** ai-engineering-from-scratch -> Phase 0 (Setup & Tooling), Lesson 11
**Type:** Learn | **Languages:** -- | **Time:** ~30 minutes
**Prerequisites:** Phase 0, Lesson 01

This lesson is a survival guide for remote Linux machines — cloud GPU boxes, Lambda instances, EC2. Run the code cells to see visualizations of the file system, permissions, and disk management.


# The Problem: You Land in Ubuntu With No GUI

---


You develop on macOS or Windows. But the moment you SSH into a cloud GPU box, rent a Lambda instance, or spin up an EC2 machine, you land in Ubuntu. The terminal is your only interface. There is no Finder, no Explorer, no GUI.

If you can't navigate the file system, install packages, and manage processes from the command line, you're stuck paying for idle GPU hours while googling "how to unzip a file in Linux."

This is a survival guide. It covers exactly what you need to operate on a remote Linux machine for AI work. Nothing more.


# Visual: The Linux File System — What You Actually Touch

---


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

fig, ax = plt.subplots(figsize=(9, 5))
ax.axis('off')

dirs = [
    ('/', 'Root — everything starts here', '#2c3e50', 0.5, 0.9),
    ('~/home/your-username', 'Your files — clone repos, run training', '#3498db', 0.2, 0.7),
    ('/tmp', 'Temporary files, cleared on reboot', '#e74c3c', 0.8, 0.7),
    ('/usr', 'System programs and libraries', '#2ecc71', 0.2, 0.5),
    ('/etc', 'Config files', '#9b59b6', 0.5, 0.5),
    ('/var/log', 'Logs — check when something breaks', '#f39c12', 0.8, 0.5),
    ('/mnt or /media', 'External drives and volumes', '#e67e22', 0.2, 0.3),
    ('/proc and /sys', 'Virtual files — kernel and hardware info', '#95a5a6', 0.8, 0.3),
]

for label, desc, color, x, y in dirs:
    rect = patches.Rectangle((x-0.15, y-0.06), 0.3, 0.1, facecolor=color, edgecolor='black', linewidth=2)
    ax.add_patch(rect)
    ax.text(x, y+0.03, label, ha='center', va='center', color='white', fontweight='bold', fontsize=9)
    ax.text(x, y-0.03, desc, ha='center', va='center', fontsize=7, style='italic')

ax.text(0.5, 0.1, 'Almost everything you do happens in ~ (your home directory)', ha='center', fontsize=10, style='italic')

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_title("Linux File System Layout", fontsize=13, fontweight="bold", pad=20)
plt.tight_layout()
plt.show()


> **Your home directory is `~` or `/home/your-username`.** Almost everything you do happens here.


# Essential Commands — The 15 That Cover 95%

---


### Moving Around

```bash
pwd                         # Where am I?
ls                          # What's here?
ls -la                      # What's here, including hidden files with details?
cd /path/to/dir             # Go there
cd ~                        # Go home
cd ..                       # Go up one level
```

### Files and Directories

```bash
mkdir my-project            # Create a directory
mkdir -p a/b/c              # Create nested directories in one shot

cp file.txt backup.txt      # Copy a file
cp -r src/ src-backup/      # Copy a directory (recursive)

mv old.txt new.txt          # Rename a file
mv file.txt /tmp/           # Move a file

rm file.txt                 # Delete a file (no trash, it's gone)
rm -rf my-dir/              # Delete a directory and everything inside
```

> **`rm -rf` is permanent.** There is no undo. Double-check the path before hitting enter.


# Visual: `rm -rf` — No Trash, No Undo

---


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

fig, ax = plt.subplots(figsize=(8, 3))
ax.axis('off')

rect = patches.Rectangle((0.1, 0.3), 0.8, 0.5, facecolor='#e74c3c', edgecolor='black', linewidth=3)
ax.add_patch(rect)
ax.text(0.5, 0.6, 'rm -rf my-dir/', ha='center', va='center', color='white', fontweight='bold', fontsize=14)
ax.text(0.5, 0.4, 'Deleted permanently. No trash bin. No undo.', ha='center', va='center', color='white', fontsize=11)
ax.text(0.5, 0.25, 'Double-check the path before hitting Enter.', ha='center', va='center', color='white', fontsize=9)

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_title("The Most Dangerous Command You'll Run", fontsize=12, fontweight="bold", pad=20)
plt.tight_layout()
plt.show()


> **Always verify the path** with `ls` before running `rm -rf`.


# Reading Files and Searching

---


### Reading Files

```bash
cat file.txt                # Print entire file
head -20 file.txt           # First 20 lines
tail -20 file.txt           # Last 20 lines
tail -f log.txt             # Follow a log file in real time (Ctrl+C to stop)
less file.txt               # Scroll through a file (q to quit)
```

### Searching

```bash
grep "error" training.log           # Find lines containing "error"
grep -r "learning_rate" .           # Search all files in current directory
grep -i "cuda" config.yaml          # Case-insensitive search

find . -name "*.py"                 # Find all Python files under current dir
find . -name "*.ckpt" -size +1G     # Find checkpoint files larger than 1GB
```


# Permissions — Why You Get "Permission Denied"

---


Every file in Linux has an owner and permission bits. You'll run into this when scripts won't execute or you can't write to a directory.

```bash
ls -l train.py
# -rwxr-xr-- 1 user group 2048 Mar 19 10:00 train.py
#  ^^^             owner permissions: read, write, execute
#     ^^^          group permissions: read, execute
#        ^^        everyone else: read only
```

**Common fixes:**

```bash
chmod +x train.sh           # Make a script executable
chmod 755 deploy.sh         # Owner: full, others: read+execute
chmod 644 config.yaml       # Owner: read+write, others: read only

chown user:group file.txt   # Change who owns a file (needs sudo)
```

When something says "Permission denied," it's almost always a permissions issue. `chmod +x` or `sudo` will fix most cases.


# Visual: Reading Permission Bits

---


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

fig, ax = plt.subplots(figsize=(10, 4))
ax.axis('off')

# Permission string
ax.text(0.5, 0.7, '-rwxr-xr--', ha='center', va='center', fontsize=20, fontweight='bold', fontfamily='monospace')

# Owner
rect1 = patches.Rectangle((0.15, 0.3), 0.2, 0.2, facecolor='#3498db', edgecolor='black', linewidth=2)
ax.add_patch(rect1)
ax.text(0.25, 0.4, 'rwx', ha='center', va='center', color='white', fontweight='bold', fontsize=11)
ax.text(0.25, 0.25, 'Owner', ha='center', va='center', fontsize=9)

# Group
rect2 = patches.Rectangle((0.4, 0.3), 0.2, 0.2, facecolor='#2ecc71', edgecolor='black', linewidth=2)
ax.add_patch(rect2)
ax.text(0.5, 0.4, 'r-x', ha='center', va='center', color='white', fontweight='bold', fontsize=11)
ax.text(0.5, 0.25, 'Group', ha='center', va='center', fontsize=9)

# Others
rect3 = patches.Rectangle((0.65, 0.3), 0.2, 0.2, facecolor='#95a5a6', edgecolor='black', linewidth=2)
ax.add_patch(rect3)
ax.text(0.75, 0.4, 'r--', ha='center', va='center', color='white', fontweight='bold', fontsize=11)
ax.text(0.75, 0.25, 'Others', ha='center', va='center', fontsize=9)

ax.text(0.5, 0.1, 'r = read, w = write, x = execute', ha='center', fontsize=10, style='italic')

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_title("Linux Permission Bits: Owner / Group / Others", fontsize=12, fontweight="bold", pad=20)
plt.tight_layout()
plt.show()


> **chmod +x** adds execute permission. **chmod 755** sets owner=full, group=read+execute, others=read+execute.


# Package Management — apt

---


Ubuntu uses `apt`. This is how you install system-level software.

```bash
sudo apt update             # Refresh the package list (always do this first)
sudo apt install -y htop    # Install a package (-y skips confirmation)
sudo apt install -y build-essential  # C compiler, make, etc. Needed by many Python packages
sudo apt install -y tmux    # Terminal multiplexer (keep sessions alive after disconnect)

apt list --installed        # What's installed?
sudo apt remove htop        # Uninstall
```

**Common packages you'll install on a fresh GPU box:**

```bash
sudo apt update && sudo apt install -y \
    build-essential \
    git \
    curl \
    wget \
    tmux \
    htop \
    unzip \
    python3-venv
```


# Users and sudo

---


You're usually logged in as a regular user. Some operations need root (admin) access.

```bash
whoami                      # What user am I?
sudo command                # Run a single command as root
sudo su                     # Become root (exit to go back, use sparingly)
```

On cloud GPU instances, you're typically the only user and already have sudo access. **Don't run everything as root.** Use sudo only when needed.


# Processes and systemd

---


When your training hangs, or you need to check what's running:

```bash
htop                        # Interactive process viewer (q to quit)
ps aux | grep python        # Find running Python processes
kill 12345                  # Gracefully stop process with PID 12345
kill -9 12345               # Force kill (use when graceful doesn't work)
nvidia-smi                  # GPU processes and memory usage
```

**systemd** manages services (background daemons). You'll use it if you run inference servers:

```bash
sudo systemctl start nginx          # Start a service
sudo systemctl stop nginx           # Stop it
sudo systemctl restart nginx        # Restart it
sudo systemctl status nginx         # Check if it's running
sudo systemctl enable nginx         # Start automatically on boot
```


# Visual: Process Lifecycle — Start, Monitor, Kill

---


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

fig, ax = plt.subplots(figsize=(10, 4))
ax.axis('off')

steps = [
    ('Start process', 'python train.py', '#3498db'),
    ('Monitor', 'htop / ps aux', '#2ecc71'),
    ('Graceful stop', 'kill PID', '#f39c12'),
    ('Force stop', 'kill -9 PID', '#e74c3c'),
]

y = 0.85
for label, desc, color in steps:
    rect = patches.Rectangle((0.1, y-0.08), 0.8, 0.14, facecolor=color, edgecolor='black', linewidth=2)
    ax.add_patch(rect)
    ax.text(0.15, y, label, ha='left', va='center', color='white', fontweight='bold', fontsize=10)
    ax.text(0.85, y, desc, ha='right', va='center', fontsize=8, style='italic')
    y -= 0.15

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_title("Process Management: Start → Monitor → Stop", fontsize=12, fontweight="bold", pad=20)
plt.tight_layout()
plt.show()


> **kill** sends SIGTERM (graceful). **kill -9** sends SIGKILL (immediate, no cleanup). Try graceful first.


# Disk Space — GPU Boxes Fill Up Fast

---


GPU boxes often have limited disk space. Models and datasets fill it fast.

```bash
df -h                       # Disk usage for all mounted drives
df -h /home                 # Disk usage for /home specifically

du -sh *                    # Size of each item in current directory
du -sh ~/.cache             # Size of your cache (pip, huggingface models land here)
du -sh /data/checkpoints/   # Check how big your checkpoints are

# Find the biggest space hogs
du -h --max-depth=1 / 2>/dev/null | sort -hr | head -20
```

**Common space savers:**

```bash
# Clear pip cache
pip cache purge

# Clear apt cache
sudo apt clean

# Remove old checkpoints you don't need
rm -rf checkpoints/epoch_01/ checkpoints/epoch_02/
```


# Networking — Download, Transfer, Query

---


You'll download models, transfer files, and hit APIs from the command line.

```bash
# Download files
wget https://example.com/model.bin                   # Download a file
curl -O https://example.com/data.tar.gz              # Same thing with curl
curl -s https://api.example.com/health | python3 -m json.tool  # Hit an API, pretty-print JSON

# Transfer files between machines
scp model.bin user@remote:/data/                     # Copy file to remote machine
scp user@remote:/data/results.csv .                  # Copy file from remote to local
scp -r user@remote:/data/checkpoints/ ./local-dir/   # Copy directory

# Sync directories (faster than scp for large transfers, resumes on failure)
rsync -avz --progress ./data/ user@remote:/data/
rsync -avz --progress user@remote:/results/ ./results/
```

Use `rsync` over `scp` for anything large. It only transfers changed bytes and handles interrupted connections.


# tmux — Keep Sessions Alive When SSH Disconnects

---


When you SSH into a remote box, closing your laptop kills your training run. tmux prevents this.

```bash
tmux new -s train           # Start a new session named "train"
# ... start your training, then:
# Ctrl+B, then D            # Detach (training keeps running)

tmux ls                     # List sessions
tmux attach -t train        # Reattach to session

# Inside tmux:
# Ctrl+B, then %            # Split pane vertically
# Ctrl+B, then "            # Split pane horizontally
# Ctrl+B, then arrow keys   # Switch between panes
```

**Always run long training jobs inside tmux. Always.**


# WSL2 for Windows Users

---


If you're on Windows, WSL2 gives you a real Linux environment without dual-booting.

```bash
# In PowerShell (admin)
wsl --install -d Ubuntu-24.04

# After restart, open Ubuntu from Start menu
sudo apt update && sudo apt upgrade -y
```

WSL2 runs a real Linux kernel. Everything in this lesson works inside it. Your Windows files are at `/mnt/c/Users/YourName/` from inside WSL.

GPU passthrough works with NVIDIA drivers installed on the Windows side. Install the Windows NVIDIA driver (not the Linux one), and CUDA will be available inside WSL2.


# Gotchas: macOS to Linux

---


Things that will trip you up if you're coming from macOS:

| macOS | Linux | Notes |
|-------|-------|-------|
| `brew install` | `sudo apt install` | Different package names sometimes |
| `open file.txt` | `xdg-open file.txt` | But you won't have a GUI on a remote box. Use `cat` or `less`. |
| `pbcopy` / `pbpaste` | Not available | Pipe to/from clipboard doesn't exist over SSH. |
| `~/.zshrc` | `~/.bashrc` | macOS defaults to zsh. Most Linux servers use bash. |
| `/opt/homebrew/` | `/usr/bin/`, `/usr/local/bin/` | Binaries live in different places. |
| `sed -i '' 's/a/b/' file` | `sed -i 's/a/b/' file` | macOS sed needs an empty string after `-i`. Linux does not. |
| Case-insensitive filesystem | Case-sensitive filesystem | `Model.py` and `model.py` are two different files on Linux. |
| Line endings `\n` | Line endings `\n` | Same. But Windows uses `\r\n`, which breaks bash scripts. Run `dos2unix` to fix. |


# Quick Reference Card

---


```
Navigation:     pwd, ls, cd, find
Files:          cp, mv, rm, mkdir, cat, head, tail, less
Search:         grep, find
Permissions:    chmod, chown, sudo
Packages:       apt update, apt install
Processes:      htop, ps, kill, nvidia-smi
Services:       systemctl start/stop/restart/status
Disk:           df -h, du -sh
Network:        curl, wget, scp, rsync
Sessions:       tmux new/attach/detach
```


# Quiz Review

---


Try answering these before reading the explanations:

**Q1:** What does the '~' symbol represent in a Linux file path?
> **Explanation:** The tilde (~) is a shortcut for the current user's home directory, typically /home/username on Linux.

**Q2:** What is the purpose of 'sudo' before a command?
> **Explanation:** sudo (superuser do) temporarily elevates your privileges to root level for a single command.

**Q3:** You get 'Permission denied' when trying to run a shell script. What command fixes this?
> **Explanation:** `chmod +x` adds execute permission to the file.

**Q4:** On a remote GPU box, your training data fills the disk. Which command shows the largest directories?
> **Explanation:** `du -h --max-depth=1 / | sort -hr | head -20` shows disk usage per top-level directory, sorted by size.

**Q5:** What is a key difference between macOS and Linux versions of `sed -i`?
> **Explanation:** macOS sed requires an empty string argument after `-i` (`sed -i ""`), while Linux does not.


# Practice Exercises

---


Do these in order:

**Exercise 1:** SSH into any Linux machine (or open WSL2) and navigate to your home directory. Create a project folder, create three empty files inside it with `touch`, then list them with `ls -la`.

**Exercise 2:** Install `htop` with apt, run it, and identify which process is using the most memory.

**Exercise 3:** Start a tmux session, run `sleep 300` inside it, detach, list sessions, and reattach.

**Exercise 4:** Use `df -h` to check available disk space, then use `du -sh ~/.cache/*` to find what's taking up space.

**Exercise 5:** Transfer a file from your local machine to a remote one using `scp`, then do the same with `rsync`.


# Key Terms

---


| Term | What people say | What it actually means |
|------|----------------|----------------------|
| Root directory | "The C: drive equivalent" | `/` — the top of the Linux file system. Everything starts here. |
| sudo | "Run as admin" | Temporarily elevates privileges to root for a single command |
| chmod | "Change permissions" | Modifies read/write/execute bits on a file or directory |
| apt | "Ubuntu's app store" | The package manager for Debian/Ubuntu Linux systems |
| systemd | "Service manager" | Manages background daemons and system services |
| tmux | "Terminal that survives disconnect" | Lets you detach from and reattach to terminal sessions |


# Next Steps

---


When you're comfortable on a Linux command line, continue to:
**Lesson 12: Debugging and Profiling**

Your `ai-engineering-from-scratch` learning directory is **completely untouched** — these notes live separately in this folder.
